In [1]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

def load_npz(path):
    data = np.load(path, allow_pickle=True)
    X = data["data"]
    y = data["labels"]

    X = np.array(X)
    y = np.array(y)

    return X, y


# ===== load =====
X_src, y_src = load_npz("./datasets/tor_100w_2500tr.npz")
X_tgt, y_tgt = load_npz("./datasets/tor_200w-100w_2500.npz")

# 标签要从str映射为整数
y_src = np.array(y_src).astype(str)
y_tgt = np.array(y_tgt).astype(str)
# ⭐关键：用 union，而不是只用 source
all_classes = set(y_src) | set(y_tgt)

cls2id = {c: i for i, c in enumerate(sorted(all_classes))}

y_src = np.array([cls2id[c] for c in y_src])
y_tgt = np.array([cls2id[c] for c in y_tgt])

In [5]:
class WFDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]

        # 保证 shape = (1, 5000)
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)

        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y

batch_size = 64

src_loader = DataLoader(
    WFDataset(X_src, y_src),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

tgt_loader = DataLoader(
    WFDataset(X_tgt, y_tgt),
    batch_size=batch_size,
    shuffle=False
)

## DF模型

In [11]:
import torch.nn as nn
from DF import *
model = DF(num_classes=len(unique_classes)).cuda()
print("unique labels (src):", len(set(y_src)))
print("unique labels (tgt):", len(set(y_tgt)))

print("max src:", np.max(y_src))
print("max tgt:", np.max(y_tgt))
print("min src:", np.min(y_src))
print("min tgt:", np.min(y_tgt))

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 训练代码

In [9]:
import torch
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in src_loader:
        x, y = x.cuda(), y.cuda()

        logits, feat = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}] loss = {total_loss/len(src_loader):.4f}")

../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [9,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [12,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [13,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [14,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [19,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_lo

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 测试

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.cuda(), y.cuda()

            logits, _ = model(x)
            pred = torch.argmax(logits, dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    print("Target Accuracy:", acc)
    return acc


evaluate(model, tgt_loader)

In [10]:
print("num_classes:", logits.shape[1])
print("y min:", y.min().item())
print("y max:", y.max().item())

num_classes: 100


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
